In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from prometheus_client import push_to_gateway
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(12046)

/Users/hujinjia/PycharmProjects/JupyterProject/.venv1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
batch_size = 1000
learning_rate = 0.001
eval_iters = 10
sequence_len = 64
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("使用MPS！")
else:
    device = torch.device("cpu")
    print("使用CPU")

使用MPS！


In [5]:
data_path = "/Users/hujinjia/PycharmProjects/JupyterProject/python/python/final/jsonl/train/*.jsonl.gz"
datasets = load_dataset('json', data_files=data_path)
datasets = datasets['train'].filter(lambda x: 'apache/spark' in x['repo'])

In [6]:
class CharTokenizer:
    def __init__(self, data, end_ind=0):
        chars = sorted(list(set(''.join(data))))
        self.char2ind = {s: i + 1 for i, s in
                         enumerate(chars)}  # 创建 {字符: 索引} 的映射。self.char2ind = {'a': 2, 'b': 3, 'c': 4}
        self.char2ind['<|e|>'] = end_ind
        self.ind2char = {v: k for k, v in
                         self.char2ind.items()}  # 创建从索引到字符的反向映射，便于解码。self.ind2chat = {0: '<|b|>', 1: '<|e|>', 2: 'a', 3: 'b', 4: 'c'}
        self.end_ind = end_ind

    def encode(self, x):
        return [self.char2ind[i] for i in x]

    def decode(self, x):
        if isinstance(x, int):
            return self.ind2char[x]
        else:
            return [self.ind2char[i] for i in x]


tokenizer = CharTokenizer(datasets['original_string'])
#测试
test_str = 'def f(x)'
re = tokenizer.encode(test_str)
print(re)
''.join(tokenizer.decode(range(len(tokenizer.char2ind))))

[70, 71, 72, 2, 72, 10, 90, 11]


'<|e|>\n !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~ö'

In [7]:
def process(data, tokenizer, sequence_len):
    text = data['original_string']
    inputs, labels = [], []
    for t in text:
        enc = tokenizer.encode(t)
        enc += [tokenizer.end_ind]
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i:i + sequence_len])
            labels.append(enc[i + 1:i + 1 + sequence_len])
    return {'inputs': inputs, 'labels': labels}


tokenized = datasets.train_test_split(test_size=0.1, seed=1024, shuffle=True)
f = lambda x: process(x, tokenizer, sequence_len=64)
tokenized = tokenized.map(f, batched=True, remove_columns=datasets.column_names)
tokenized.set_format(type='torch', device=device)

Map: 100%|██████████| 69/69 [00:00<00:00, 124.07 examples/s]


In [8]:
train_loader = DataLoader(tokenized['train'], batch_size=1000, shuffle=True)
test_loader = DataLoader(tokenized['test'], batch_size=1000, shuffle=True)
next(iter(train_loader))

{'inputs': tensor([[ 2, 84, 67,  ...,  2, 86, 91],
         [69, 74, 71,  ...,  2,  2,  2],
         [71,  2, 12,  ..., 85, 71, 78],
         ...,
         [ 2,  2,  2,  ..., 71, 16,  1],
         [67, 79, 71,  ...,  2, 84, 71],
         [ 2,  2,  2,  ..., 79,  2, 71]], device='mps:0'),
 'labels': tensor([[84, 67, 75,  ..., 86, 91, 82],
         [74, 71, 16,  ...,  2,  2,  2],
         [ 2, 12, 71,  ..., 71, 78, 91],
         ...,
         [ 2,  2,  2,  ..., 16,  1,  1],
         [79, 71,  2,  ..., 84, 71, 85],
         [ 2,  2,  2,  ...,  2, 71, 80]], device='mps:0')}

In [9]:
@torch.no_grad()
# 生成字
def generate(model, context,tokenizer,max_new_tokens=300):
    # context:(B,T), B =1
    # out = []
    out = context.tolist()[0] #将背景放到输入中
    model.eval()
    for _ in range(max_new_tokens):
        logits = model(context) #(B,T,VS) -> (1,T,98)
        probs = F.softmax(logits[:,-1,:], dim=-1) # 取最后一个logits (1, 98)
        #随机生成文本
        ix = torch.multinomial(probs, num_samples=1)  #(1,1)
        # 更新背景
        context = torch.concat((context, ix), dim=-1)
        out.append(ix.item())
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [17]:
class LSTMCell(nn.Module):
    def __init__(self,input_size,hidden_size):
        super(LSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        combined_size = hidden_size + input_size
        self.forget_gate = nn.Linear(combined_size, hidden_size)
        self.in_gate = nn.Linear(combined_size, hidden_size)
        self.new_cell_state = nn.Linear(combined_size, hidden_size)
        self.out_gate = nn.Linear(combined_size, hidden_size)

    def forward(self,input,state=None):
        # input:(B,I) I表示input_size
        # state是隐藏状态和细胞状态的排列(假设他俩形状一样） state：(B,H)（B,H)
        B = input.size(0)
        if state is None:
            state = self.init_state(B, input.device)
        hs, cs = state
        combined = torch.concat((input,hs), dim = -1)
        # 细胞状态更新
        ingate = F.sigmoid(self.in_gate(combined))
        forgetgate = F.sigmoid(self.forget_gate(combined))
        ncs = F.tanh(self.new_cell_state(combined)) # 新细胞状态
        cs = (cs * forgetgate) + (ingate * ncs) # 更新细胞状态
        # 隐藏状态更新
        outgate = F.sigmoid(self.out_gate(combined))
        hs = F.tanh(cs) * outgate
        return hs, cs

    def init_state(self, B, device):
        # device = device 确保设备一致性
        hs = torch.zeros((B, self.hidden_size), device=device)
        cs = torch.zeros((B, self.hidden_size), device=device)
        return hs, cs

In [18]:
# 测试上面
l_cell = LSTMCell(3,4)
x = torch.randn(5,3)
a,b = l_cell(x)
print(a.shape)
print(b.shape)

torch.Size([5, 4])
torch.Size([5, 4])


In [19]:
class LSTM(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.cell = LSTMCell(input_size, hidden_size)

    def init_hidden(self, B,  device):
        return torch.zeros((B, self.hidden_size), device=device)

    def forward(self, input, state=None):
        # input:  (B,T,C)
        # state: (B,  H), (B, H)
        # out:    (B,T,H)
        B, T, C = input.shape
        re = []
        for i in range(T):
            state = self.cell(input[: ,i, :], state)
            re.append(state[0])
        return torch.stack(re, dim=1)   #(B,T,H)

In [20]:
class CharLSTM(nn.Module):
    def __init__(self, vs):
        # vocabulary size
        super().__init__()
        self.emb_size = 256 # 词嵌入维度
        self.hidden_size = 128
        self.emb = nn.Embedding(vs, self.emb_size) #将字符索引转换为256维的向量表示
        self.dp = nn.Dropout(0.4)
        self.lstm1 = LSTM(self.emb_size, self.hidden_size)
        self.ln1 = nn.LayerNorm(self.hidden_size)
        self.lstm2 = LSTM(self.hidden_size, self.hidden_size)
        self.ln2 = nn.LayerNorm(self.hidden_size)
        self.lstm3 = LSTM(self.hidden_size, self.hidden_size)
        self.ln3 = nn.LayerNorm(self.hidden_size)
        self.lm = nn.Linear(self.hidden_size, vs) #将LSTM输出映射回词汇表大小的概率分布

    def forward(self, x):
        # x:(B , T) B: batch size（批量大小），一次处理多少个独立的序列，T: sequence length（序列长度）
        embedding = self.emb(x) # (B, T, C)： (batch_size, seq_len) → (batch_size, seq_len, 256)
        h = self.ln1(self.dp(self.lstm1(embedding))) # (B, T, H)
        h = self.ln2(self.dp(self.lstm2(h)))
        h = self.ln3(self.dp(self.lstm3(h)))
        output = self.lm(h)
        return output

In [22]:
c_model = CharLSTM(len(tokenizer.char2ind)).to(device)
c_model

CharLSTM(
  (emb): Embedding(98, 256)
  (dp): Dropout(p=0.4, inplace=False)
  (lstm1): LSTM(
    (cell): LSTMCell(
      (forget_gate): Linear(in_features=384, out_features=128, bias=True)
      (in_gate): Linear(in_features=384, out_features=128, bias=True)
      (new_cell_state): Linear(in_features=384, out_features=128, bias=True)
      (out_gate): Linear(in_features=384, out_features=128, bias=True)
    )
  )
  (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lstm2): LSTM(
    (cell): LSTMCell(
      (forget_gate): Linear(in_features=256, out_features=128, bias=True)
      (in_gate): Linear(in_features=256, out_features=128, bias=True)
      (new_cell_state): Linear(in_features=256, out_features=128, bias=True)
      (out_gate): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lstm3): LSTM(
    (cell): LSTMCell(
      (forget_gate): Linear(in_features=256, out_features=128, bias=True)
   

In [23]:
context = torch.tensor( tokenizer.encode('def'),device = device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context,tokenizer))))

defC&JiLc6f5Lz/T(>yDZ$L\=H<|e|>


In [24]:
def estimate_loss(model):
    re = {} #创建一个空字典，用于存储训练损失和测试损失,最终返回的结构：{'train': 0.123, 'test': 0.456}
    model.eval() #将训练模式切换为评估模式
    train_loss = _loss(model, train_loader)
    test_loss = _loss(model, test_loader)
    re['train'] = train_loss
    re['test'] = test_loss
    model.train()  #切换回训练模式
    return re

@torch.no_grad()
def _loss(model, data_loader):
    #计算模型在不同数据集下面的评估指标
    loss = []
    data_iter = iter(data_loader)
    # 随机使用多个批量数据来评估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None) #如果没有数据，返回 None
        if data is None: # 如果当前迭代器没有数据了
            data_iter = iter(data_loader) # 重新创建迭代器
            data = next(data_iter, None) # 获取第一个 batch
        inputs, labels = data['inputs'], data['labels'] # (B, T)
        logits = model(inputs)                          # (B, T, vs)
        # 官网交叉熵需要的纬度（B ，VS ，...)
        loss.append(F.cross_entropy(logits.transpose(-2,-1), labels).item()) # item() 将损失值(张量）转换为 Python 数字
    return torch.tensor(loss).mean().item()

estimate_loss(c_model)

{'train': 4.751618385314941, 'test': 4.745863437652588}

In [25]:
def train_model(model,optimizer,epochs=10):
    lossi = [] # 记录模型在训练集上的模型损失
    for epoch in range(epochs):
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data['inputs'], data['labels'] # (B, T)
            optimizer.zero_grad()
            logits = model(inputs)                          # (B ,T vs)
            loss = F.cross_entropy(logits.transpose(-2,-1), labels)          # 如上
            lossi.append(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats['train']:.4f}'
        test_loss = f'test loss {stats['test']:.4f}'
        print(f'epoch {epoch:>2}: {train_loss}, {test_loss}')
    return lossi

In [26]:
l = train_model(c_model, optim.Adam(c_model.parameters(), lr=learning_rate))

epoch  0: train loss 1.2761, test loss 1.4211
epoch  1: train loss 1.1021, test loss 1.2936
epoch  2: train loss 1.0315, test loss 1.2474
epoch  3: train loss 0.9992, test loss 1.2216
epoch  4: train loss 0.9685, test loss 1.2178
epoch  5: train loss 0.9438, test loss 1.1995
epoch  6: train loss 0.9270, test loss 1.1879
epoch  7: train loss 0.9145, test loss 1.1931
epoch  8: train loss 0.9044, test loss 1.1857
epoch  9: train loss 0.8916, test loss 1.1812


In [27]:
context = torch.tensor( tokenizer.encode('def'),device = device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context,tokenizer))))

def textFile, length=0, self.numLocation=None, name=None, nullValue=None,
                                                                computed="orgers_type", self._sc, maxBins=None, key=None, maxColumns=None,
            columnNameOfCorruptRecord=None, partitionFunc=None, name=None,
               
